In [17]:
import pandas as pd
from sklearn.model_selection import train_test_split


In [18]:
df = pd.read_csv("../data/processed_telco_churn.csv")

In [19]:
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,OnlineSecurity,OnlineBackup,DeviceProtection,...,MonthlyCharges,TotalCharges,Churn,InternetService_Fiber optic,InternetService_No,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,0,1,0,1,0,0,0,1,0,...,29.85,29.85,0,False,False,False,False,False,True,False
1,1,0,0,0,34,1,0,1,0,1,...,56.95,1889.50,0,False,False,True,False,False,False,True
2,1,0,0,0,2,1,0,1,1,0,...,53.85,108.15,1,False,False,False,False,False,False,True
3,1,0,0,0,45,0,0,1,0,1,...,42.30,1840.75,0,False,False,True,False,False,False,False
4,0,0,0,0,2,1,0,0,0,0,...,70.70,151.65,1,True,False,False,False,False,True,False


In [20]:
X = df.drop("Churn", axis=1)

y = df["Churn"]

In [21]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y, 
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [22]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
results = []

In [23]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lr = LogisticRegression(
    random_state=42,
    class_weight="balanced",
)

lr.fit(X_train_scaled, y_train)


pred_lr = lr.predict(X_test_scaled)
accuracy_score(y_test, pred_lr)

from sklearn.metrics import confusion_matrix

pred_lr = lr.predict(X_test_scaled)

print(confusion_matrix(y_test, pred_lr))
print(classification_report(y_test, pred_lr))

results.append({
    "Model": "Logistic Regression",
    "Accuracy": accuracy_score(y_test, pred_lr),
    "Precision": precision_score(y_test, pred_lr),
    "Recall": recall_score(y_test, pred_lr),
    "F1 Score": f1_score(y_test, pred_lr)
})

[[724 309]
 [ 76 298]]
              precision    recall  f1-score   support

           0       0.91      0.70      0.79      1033
           1       0.49      0.80      0.61       374

    accuracy                           0.73      1407
   macro avg       0.70      0.75      0.70      1407
weighted avg       0.79      0.73      0.74      1407



In [24]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(random_state=42, class_weight="balanced")

dt.fit(X_train, y_train)

pred_dt = dt.predict(X_test)
accuracy_score(y_test, pred_dt)

results.append({
    "Model": "Decision Tree",
    "Accuracy": accuracy_score(y_test, pred_dt),
    "Precision": precision_score(y_test, pred_dt),
    "Recall": recall_score(y_test, pred_dt),
    "F1 Score": f1_score(y_test, pred_dt)
})

In [25]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(random_state=42, class_weight="balanced")

rf.fit(X_train, y_train)

pred_rf = rf.predict(X_test)
accuracy_score(y_test, pred_rf)

results.append({
    "Model": "Random Forest",
    "Accuracy": accuracy_score(y_test, pred_rf),
    "Precision": precision_score(y_test, pred_rf),
    "Recall": recall_score(y_test, pred_rf),
    "F1 Score": f1_score(y_test, pred_rf)
})

In [26]:
from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(random_state=42)

gb.fit(X_train, y_train)

pred_gb = gb.predict(X_test)

accuracy_score(y_test, pred_gb)

results.append({
    "Model": "Gradient Boosting",
    "Accuracy": accuracy_score(y_test, pred_gb),
    "Precision": precision_score(y_test, pred_gb),
    "Recall": recall_score(y_test, pred_gb),
    "F1 Score": f1_score(y_test, pred_gb)
})

In [27]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    random_state=42,
    eval_metric="logloss"
)

xgb.fit(X_train, y_train)

pred_xgb = xgb.predict(X_test)

accuracy_score(y_test, pred_xgb)

results.append({
    "Model": "XGBoost",
    "Accuracy": accuracy_score(y_test, pred_xgb),
    "Precision": precision_score(y_test, pred_xgb),
    "Recall": recall_score(y_test, pred_xgb),
    "F1 Score": f1_score(y_test, pred_xgb)
})

In [28]:
results_df = pd.DataFrame(results)
results_df.sort_values(
    by="Accuracy",
    ascending=False
)

,Model,Accuracy,Precision,Recall,F1 Score
3,Gradient Boosting,0.795309,0.637821,0.532086,0.580175
2,Random Forest,0.786780,0.627586,0.486631,0.548193
4,XGBoost,0.778252,0.589080,0.548128,0.567867
0,Logistic Regression,0.726368,0.490939,0.796791,0.607543
1,Decision Tree,0.724947,0.483117,0.497326,0.490119


In [29]:
# Hyper Parameter Tuning
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

In [30]:
rf = RandomForestClassifier(random_state=42)

param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [5, 10, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

grid_rf = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1
)

grid_rf.fit(X_train, y_train) 

best_rf = grid_rf.best_estimator_

pred_rf_tuned = best_rf.predict(X_test)

rf_tuned_acc = accuracy_score(y_test, pred_rf_tuned)

rf_tuned_results = pd.DataFrame({
    "Model": ["Random Forest (Tuned)"],
    "Accuracy": [rf_tuned_acc],
    "Precision": [precision_score(y_test, pred_rf_tuned)],
    "Recall": [recall_score(y_test, pred_rf_tuned)],
    "F1 Score": [f1_score(y_test, pred_rf_tuned)],
    "Best Parameters": [grid_rf.best_params_]
})

rf_tuned_results

,Model,Accuracy,Precision,Recall,F1 Score,Best Parameters
0,Random Forest (Tuned),0.797441,0.652921,0.508021,0.571429,"{'max_depth': 20, 'min_samples_leaf': 2, 'min_..."


In [32]:
from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(random_state=42)

param_grid = {
    "n_estimators": [100, 200, 500],
    "learning_rate": [0.01, 0.05, 0.1],
    "max_depth": [3, 5, 7]
}

grid_gb = GridSearchCV(
    estimator=gb,
    param_grid=param_grid,
    scoring="f1",
    cv=5,
    n_jobs=-1
)

grid_gb.fit(X_train, y_train)

best_gb = grid_gb.best_estimator_

pred_gb = best_gb.predict(X_test)

gb_tuned_results = pd.DataFrame({
    "Model": ["Gradient Boosting (Tuned)"],
    "Accuracy": [accuracy_score(y_test, pred_gb)],
    "Precision": [precision_score(y_test, pred_gb)],
    "Recall": [recall_score(y_test, pred_gb)],
    "F1 Score": [f1_score(y_test, pred_gb)],
    "Best Parameters": [grid_gb.best_params_]
})

gb_tuned_results

,Model,Accuracy,Precision,Recall,F1 Score,Best Parameters
0,Gradient Boosting (Tuned),0.793888,0.632911,0.534759,0.57971,"{'learning_rate': 0.1, 'max_depth': 3, 'n_esti..."


In [34]:
xgb = XGBClassifier(
    random_state=42,
    eval_metric="logloss"
)

param_grid = {
    "n_estimators": [100, 200, 300],
    "learning_rate": [0.01, 0.05, 0.1],
    "max_depth": [3, 5, 7],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0]
}

grid_xgb = GridSearchCV(
    estimator=xgb,
    param_grid=param_grid,
    scoring="f1",
    cv=5,
    n_jobs=-1
)

grid_xgb.fit(X_train, y_train)

print(grid_xgb.best_params_)

best_xgb = grid_xgb.best_estimator_

pred_xgb = best_xgb.predict(X_test)

xgb_tuned_results = pd.DataFrame({
    "Model": ["XGBoost (Tuned)"],
    "Accuracy": [accuracy_score(y_test, pred_xgb)],
    "Precision": [precision_score(y_test, pred_xgb)],
    "Recall": [recall_score(y_test, pred_xgb)],
    "F1 Score": [f1_score(y_test, pred_xgb)],
    "Best Parameters": [grid_xgb.best_params_]
})

xgb_tuned_results


{'colsample_bytree': 0.8, 'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 200, 'subsample': 0.8}


,Model,Accuracy,Precision,Recall,F1 Score,Best Parameters
0,XGBoost (Tuned),0.790334,0.628664,0.516043,0.566814,"{'colsample_bytree': 0.8, 'learning_rate': 0.0..."


In [35]:
final_results = pd.concat([
    rf_tuned_results,
    gb_tuned_results,
    xgb_tuned_results
], ignore_index=True)

final_results

,Model,Accuracy,Precision,Recall,F1 Score,Best Parameters
0,Random Forest (Tuned),0.797441,0.652921,0.508021,0.571429,"{'max_depth': 20, 'min_samples_leaf': 2, 'min_..."
1,Gradient Boosting (Tuned),0.793888,0.632911,0.534759,0.579710,"{'learning_rate': 0.1, 'max_depth': 3, 'n_esti..."
2,XGBoost (Tuned),0.790334,0.628664,0.516043,0.566814,"{'colsample_bytree': 0.8, 'learning_rate': 0.0..."


In [36]:
coef = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": lr.coef_[0]
})

coef.sort_values(
    "Coefficient",
    ascending=False
)

,Feature,Coefficient
16,InternetService_Fiber optic,0.718350
15,TotalCharges,0.607310
11,StreamingTV,0.248779
12,StreamingMovies,0.242041
6,MultipleLines,0.197487
21,PaymentMethod_Electronic check,0.194695
13,PaperlessBilling,0.124488
1,SeniorCitizen,0.075268
9,DeviceProtection,0.064351
20,PaymentMethod_Credit card (automatic),0.030008
